<div style="font-family:Inter,ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #d7e1dc;border-left:5px solid #1d6b5a;border-radius:8px;background:#ffffff;color:#17211f;padding:22px;margin:12px 0 18px;line-height:1.55;overflow-wrap:anywhere;word-break:break-word;box-sizing:border-box;">
  <div style="display:flex;flex-wrap:wrap;gap:8px;margin-bottom:12px;">
    <span style="background:#e2f4ea;color:#0f5c36;border:1px solid #abd5c4;border-radius:999px;padding:5px 10px;font-size:12px;font-weight:850;line-height:1.25;">Kaggle Agents for Good</span>
    <span style="background:#eef6f2;color:#203b35;border:1px solid #c9d9d2;border-radius:999px;padding:5px 10px;font-size:12px;font-weight:850;line-height:1.25;">deterministic demo</span>
    <span style="background:#eef6f2;color:#203b35;border:1px solid #c9d9d2;border-radius:999px;padding:5px 10px;font-size:12px;font-weight:850;line-height:1.25;">no API key required</span>
  </div>
  <h1 style="margin:0 0 8px;font-size:34px;line-height:1.12;letter-spacing:0;color:#10231f;font-weight:900;">Sisyphus Watch</h1>
  <p style="font-size:18px;line-height:1.48;margin:0 0 14px;color:#253b36;font-weight:650;max-width:860px;">A notebook-safe analytical briefing for claim-version-control and epistemic separation in public-interest information.</p>
  <div style="display:flex;flex-wrap:wrap;gap:12px;align-items:stretch;">
    <div style="flex:1 1 18rem;min-width:0;border:1px solid #d7e1dc;background:#fbfcfa;border-radius:8px;padding:13px;">
      <div style="font-size:12px;font-weight:850;text-transform:uppercase;color:#6a4a07;">Problem</div>
      <p style="margin:6px 0 0;font-size:14px;">Public claims, later evidence, interpretations, and current judgment often collapse into one misleading summary.</p>
    </div>
    <div style="flex:1 1 18rem;min-width:0;border:1px solid #abd5c4;background:#f7fbf8;border-radius:8px;padding:13px;">
      <div style="font-size:12px;font-weight:850;text-transform:uppercase;color:#1d6b5a;">Sisyphus Watch</div>
      <p style="margin:6px 0 0;font-size:14px;font-weight:750;">Findings -> claims -> timeline -> drift -> graph -> revision proposal -> reviewer and agent packets.</p>
    </div>
  </div>
</div>


## Submission Summary

Sisyphus Watch is a claim-version-control and epistemic-separation agent demo for public-interest information. The notebook explains the product once at the top, then lets the functional demo flow unfold with minimal interruption.


## Product Brief and Review Map

Here is what this is. Here is how to read it. Then watch it work.


In [ ]:
from pathlib import Path
import json
import importlib.util
import sys

try:
    from IPython.display import FileLink, HTML, Markdown, display
except ModuleNotFoundError:
    class HTML(str):
        pass

    class Markdown(str):
        pass

    FileLink = None

    def display(obj):
        text = str(obj)
        print(text[:1200] + ("..." if len(text) > 1200 else ""))


def _candidate_project_roots(start=None):
    seen = set()

    def add(path: Path):
        resolved = path.expanduser().resolve()
        if resolved.is_file():
            resolved = resolved.parent
        for candidate in [resolved, *resolved.parents]:
            if candidate not in seen:
                seen.add(candidate)
                yield candidate

    if start is not None:
        yield from add(start)

    yield from add(Path.cwd())

    kaggle_working = Path("/kaggle/working")
    if kaggle_working.exists() and kaggle_working not in seen:
        seen.add(kaggle_working)
        yield kaggle_working

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for module_path in kaggle_input.glob("**/src/sisyphus_watch_demo.py"):
            root = module_path.parents[1].resolve()
            if root not in seen:
                seen.add(root)
                yield root


def _load_sisyphus_demo_module():
    for root in _candidate_project_roots(Path.cwd()):
        module_path = root / "src" / "sisyphus_watch_demo.py"
        if module_path.exists():
            src_path = str(root / "src")
            if src_path not in sys.path:
                sys.path.insert(0, src_path)
            spec = importlib.util.spec_from_file_location("sisyphus_watch_demo", module_path)
            if spec is None or spec.loader is None:
                continue
            module = importlib.util.module_from_spec(spec)
            sys.modules["sisyphus_watch_demo"] = module
            spec.loader.exec_module(module)
            return module

    raise FileNotFoundError(
        "Could not locate src/sisyphus_watch_demo.py. Expected a repo root, "
        "notebook inside notebooks/, or a Kaggle input dataset containing the project folder."
    )


demo = _load_sisyphus_demo_module()
import sisyphus_watch_demo

print(f"sisyphus_watch_demo.__file__={sisyphus_watch_demo.__file__}")

_REQUIRED_SISYPHUS_SYMBOLS = [
    "render_product_brief_html",
    "render_review_map_html",
    "build_surface_model",
    "render_agent_contact_surface_html",
]
_missing_sisyphus_symbols = [
    name for name in _REQUIRED_SISYPHUS_SYMBOLS if not hasattr(sisyphus_watch_demo, name)
]
if _missing_sisyphus_symbols:
    raise RuntimeError(
        "The attached Kaggle dataset contains an older src/sisyphus_watch_demo.py. "
        "Re-upload or reattach the dataset generated from the same GitHub commit as this notebook. "
        f"Missing symbols: {', '.join(_missing_sisyphus_symbols)}. "
        f"Resolved module path: {sisyphus_watch_demo.__file__}"
    )

PROJECT_ROOT = sisyphus_watch_demo.find_project_root(Path.cwd())
src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from sisyphus_watch_demo import (
    build_deterministic_discovery_packet,
    build_guided_flow_summary,
    build_surface_model,
    build_user_problem_packet,
    build_agent_packet,
    fallback_to_demo_records,
    filter_sources_for_card,
    find_project_root,
    get_news_cards,
    get_evidence_patch_for_scenario,
    load_demo_sources,
    load_evidence_patches,
    load_scenario_authoring_template,
    maybe_run_live_extraction,
    render_agent_capability_strip_html,
    render_agent_contact_surface_html,
    render_user_problem_card_html,
    render_plain_summary_vs_sisyphus_html,
    render_product_brief_html,
    render_guided_flow_html,
    render_discovery_packet_html,
    render_judge_quickstart_html,
    render_run_status_html,
    render_review_map_html,
    render_course_concepts_html,
    render_export_artifacts_overview_html,
    render_submission_readiness_html,
    render_surface_model_html,
    maybe_run_google_ai_discovery,
    render_branch_view_html,
    render_agent_workflow_trace_html,
    render_kaggle_midcheck_summary_html,
    render_reviewer_dashboard_html,
    render_card_html,
    render_json_export,
    render_epistemic_layers_html,
    render_evaluation_summary_html,
    render_quality_checks_html,
    render_sources_table_html,
    render_version_timeline_html,
    render_claim_drift_html,
    render_claim_graph_html,
    render_graph_query_preview_html,
    render_reviewer_presets_html,
    render_revision_proposal_html,
    render_revision_comparison_html,
    render_scenario_authoring_preview_html,
    run_quality_checks,
    select_news_card,
    to_all_cards_jsonl,
    to_jsonl,
    write_export_artifacts,
)

from sisyphus_watch_adk_demo import (
    build_adk_capability_manifest,
    run_sisyphus_adk_demo,
)
from sisyphus_watch_mcp_server import build_mcp_capability_manifest

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
USER_PROBLEM = "During a severe city heatwave, did the announced cooling centers remain accessible after residents reported closures, limited hours, signage problems, and transport barriers?"
SCENARIO_ID = "city_heatwave_cooling_centers"
RUN_GOOGLE_DISCOVERY = False
RUN_LIVE_MODE = False

source_records = load_demo_sources()

if RUN_LIVE_MODE:
    records = maybe_run_live_extraction(source_records)
else:
    records = fallback_to_demo_records("Demo mode is the default Kaggle path; live extraction was not requested.")

all_news_cards = get_news_cards(records)
news_card = select_news_card(records, SCENARIO_ID)
selected_source_records = filter_sources_for_card(source_records, news_card)
agent_packet = build_agent_packet(news_card)
evidence_patches = load_evidence_patches()
evidence_patch = get_evidence_patch_for_scenario(evidence_patches, news_card["scenario_id"])

problem_packet = build_user_problem_packet(
    USER_PROBLEM,
    SCENARIO_ID,
    "google_ai_discovery" if RUN_GOOGLE_DISCOVERY else "deterministic_fixture_discovery",
)

if RUN_GOOGLE_DISCOVERY:
    discovery_packet = maybe_run_google_ai_discovery(USER_PROBLEM, selected_source_records, SCENARIO_ID)
else:
    discovery_packet = build_deterministic_discovery_packet(USER_PROBLEM, selected_source_records, SCENARIO_ID)

guided_flow_summary = build_guided_flow_summary(
    news_card,
    selected_source_records,
    discovery_packet=discovery_packet,
    evidence_patch=evidence_patch,
)

adk_manifest = build_adk_capability_manifest()
adk_demo_trace = run_sisyphus_adk_demo(SCENARIO_ID, USER_PROBLEM)
mcp_manifest = build_mcp_capability_manifest()

fallback_reasons = []
if RUN_LIVE_MODE and records.get("fallback_reason"):
    fallback_reasons.append("Record path: " + records["fallback_reason"])
if discovery_packet.get("fallback_reason"):
    fallback_reasons.append("Discovery path: " + discovery_packet["fallback_reason"])

run_status = {
    "run_google_discovery": RUN_GOOGLE_DISCOVERY,
    "run_live_mode": RUN_LIVE_MODE,
    "discovery_mode": discovery_packet.get("mode"),
    "record_mode": records.get("mode", "demo"),
    "fallback_reasons": fallback_reasons,
    "selected_card_id": news_card.get("card_id"),
    "selected_scenario_id": news_card.get("scenario_id", SCENARIO_ID),
    "available_demo_card_count": len(all_news_cards),
    "evidence_patch_available": evidence_patch is not None,
    "export_path_target": "/kaggle/working",
}

surface_model = build_surface_model(
    news_card,
    evidence_patch=evidence_patch,
    discovery_packet=discovery_packet,
    adk_manifest=adk_manifest,
    mcp_manifest=mcp_manifest,
)

display(HTML(render_product_brief_html(news_card)))
display(HTML(render_review_map_html(
    surface_model,
    run_status=run_status,
    adk_manifest=adk_manifest,
    mcp_manifest=mcp_manifest,
)))
display(HTML(render_run_status_html(run_status)))


## Two-Surface Architecture

Core State is shared. Human Review Workflow explains. Agent Contact Surface contracts.


In [ ]:
display(HTML(render_surface_model_html(surface_model)))


## Demo Flow

The walkthrough below follows the human review path first, then the agent export surface near the end.


## User Problem

The agent starts from a public-interest question.


In [ ]:
display(HTML(render_user_problem_card_html(problem_packet)))


## Discovery

Discovery prepares source candidates while keeping optional live results review-only.


In [ ]:
display(HTML(render_discovery_packet_html(discovery_packet)))


## Epistemic Separation

Findings, actor claims, actions, interpretations, and judgment stay separate.


In [ ]:
display(HTML(render_epistemic_layers_html(news_card)))


## Human Card

The canonical card turns messy public information into a readable layered record.


In [ ]:
display(HTML(render_card_html(news_card)))


## Version Timeline

The timeline tracks how the public claim changes across versions.


In [ ]:
display(HTML(render_version_timeline_html(news_card)))


## Claim Drift

Drift marks whether claims strengthened, weakened, narrowed, complicated, or remain unresolved.


In [ ]:
display(HTML(render_claim_drift_html(news_card)))


## Claim Graph

The graph exposes reusable evidence and claim relationships.


In [ ]:
display(HTML(render_claim_graph_html(news_card)))


## Evidence Patch

New evidence is reviewed as a non-mutating patch.


In [ ]:
display(HTML(render_revision_proposal_html(news_card, evidence_patch)))


## Revision Comparison

The revision preview shows what would change and what remains uncertain.


In [ ]:
display(HTML(render_revision_comparison_html(news_card, evidence_patch)))


## Course Concepts Demonstrated

The rubric evidence stays visible in a compact panel.


In [ ]:
display(HTML(render_course_concepts_html(adk_manifest, adk_demo_trace, mcp_manifest)))


## Agent Contact Surface

The agent contact surface exposes reusable JSON/JSONL/MCP artifacts.


In [ ]:
display(HTML(render_agent_contact_surface_html(surface_model, news_card, evidence_patch)))


## Downloadable Export Artifacts

Exports write the machine-readable artifacts for reuse.


In [ ]:
kaggle_output_dir = Path("/kaggle/working")
display(HTML(render_export_artifacts_overview_html(news_card, evidence_patch, "/kaggle/working")))

if kaggle_output_dir.exists():
    export_paths = write_export_artifacts(news_card, kaggle_output_dir)
    display(Markdown("Export artifacts written to `/kaggle/working`."))
    if FileLink is not None:
        for label, export_path in export_paths.items():
            display(FileLink(str(export_path), result_html_prefix=f"{label}: "))
    else:
        for label, export_path in export_paths.items():
            print(f"{label}: {export_path}")
else:
    display(Markdown("Local run: `/kaggle/working` is unavailable, so export artifact writing was skipped."))

## Evaluation

The final checks confirm the deterministic notebook path and structured outputs.


In [ ]:
checks = run_quality_checks(news_card)
display(HTML(render_submission_readiness_html(
    news_card,
    evidence_patch,
    checks,
    discovery_packet=discovery_packet,
    adk_manifest=adk_manifest,
    mcp_manifest=mcp_manifest,
)))
display(HTML(render_evaluation_summary_html(checks, news_card)))
display(HTML(render_quality_checks_html(checks)))

if not all(row["status"] == "PASS" for row in checks):
    failed = [row for row in checks if row["status"] != "PASS"]
    raise AssertionError(f"Sisyphus Watch demo checks failed: {failed}")

print("Demo mode PASS: deterministic card, human review workflow, agent contact surface, and export checks are available.")


## Kaggle Mid-Check Checklist

This readout summarizes submission readiness.


In [ ]:
display(HTML(render_kaggle_midcheck_summary_html(news_card, evidence_patch)))


## Limitations

- Sisyphus Watch is not an automatic truth oracle.
- It depends on source quality and source coverage.
- Strategic intent remains uncertain unless directly evidenced.
- Bias is labeled for review, not magically removed.
- Generated image prompts are not evidence.
- This notebook uses synthetic demo fixtures for safe, reproducible Kaggle evaluation.
- The demo does not implement live web ingestion, crawling, accounts, recommendations, a database, or a production news platform.

## Next Steps

1. Add optional JSON Schema validation with `jsonschema` when the dependency is available.
2. Package the notebook as a Kaggle Code submission with the included `data/`, `src/`, `schemas/`, and `examples/` folders.
3. Add a lightweight export command for producing JSONL from future scenarios.
